This notebook has been used to generate some first tests using GX. It is based on the GX getting started docs. It is not intended to be run externally but only for generating the json configuration which can then be executed from CRON or Airflow.

In [1]:
import great_expectations as gx
import logging

In [2]:
import os
gx_context_root_dir=os.environ['GX_CONTEXT_ROOT_DIR']

In [3]:
context = gx.get_context(context_root_dir=gx_context_root_dir)
context.list_expectation_suites()

[ExpectationSuiteIdentifier::gfw-google-827.alerts.segs_activity.2-5,
 ExpectationSuiteIdentifier::gfw-google-827.alerts.segs_activity.3-0-0,
 ExpectationSuiteIdentifier::gfw-google-827.alerts.fragments.3-0-0,
 ExpectationSuiteIdentifier::gfw-google-827.alerts.stats_daily.2-5,
 ExpectationSuiteIdentifier::gfw-google-827.alerts.stats_daily.3-0-0,
 ExpectationSuiteIdentifier::gfw-google-827.alerts.ssvids_identities_daily.2-5,
 ExpectationSuiteIdentifier::gfw-google-827.alerts.ssvids_identities_daily.3-0-0,
 ExpectationSuiteIdentifier::gfw-google-827.alerts.segs_activity_daily.2-5,
 ExpectationSuiteIdentifier::gfw-google-827.alerts.segs_activity_daily.3-0-0,
 ExpectationSuiteIdentifier::gfw-google-827.alerts.segment_vessel.2-5,
 ExpectationSuiteIdentifier::gfw-google-827.alerts.segment_vessel.3-0-0,
 ExpectationSuiteIdentifier::gfw-google-827.alerts.vessel_info.2-5,
 ExpectationSuiteIdentifier::gfw-google-827.alerts.vessel_info.3-0-0,
 ExpectationSuiteIdentifier::gfw-google-827.alerts.sat

In [4]:
import yaml

In [5]:
from datetime import date,datetime

In [6]:
logging.basicConfig(level=logging.DEBUG, force = True)

In [7]:
with open(f"{gx_context_root_dir}/datasources/datasources.yml", "r") as ymlfile:
    datasource_config = yaml.full_load(ymlfile)

In [8]:
datasource_config.get("project")

'gfw-google-827'

In [9]:
gx_project = datasource_config.get("project")
gx_datasource = context.get_datasource(gx_project)

In [10]:
gx_datasource.get_asset_names()

{'fragments-3.0.0',
 'messages-2.5',
 'messages-3.0.0',
 'satellite_timing_offsets-2.5',
 'satellite_timing_offsets-3.0.0',
 'segment_info-2.5',
 'segment_info-3.0.0',
 'segment_vessel-2.5',
 'segment_vessel-3.0.0',
 'segs_activity-2.5',
 'segs_activity-3.0.0',
 'segs_activity_daily-2.5',
 'segs_activity_daily-3.0.0',
 'ssvids_identities-2.5',
 'ssvids_identities-3.0.0',
 'ssvids_identities_daily-2.5',
 'ssvids_identities_daily-3.0.0',
 'stats_daily-2.5',
 'stats_daily-3.0.0',
 'vessel_info-2.5',
 'vessel_info-3.0.0'}

In [11]:
context.list_expectation_suite_names()

['gfw-google-827.alerts.fragments.3-0-0',
 'gfw-google-827.alerts.messages.2-5',
 'gfw-google-827.alerts.messages.3-0-0',
 'gfw-google-827.alerts.satellite_timing_offsets.2-5',
 'gfw-google-827.alerts.satellite_timing_offsets.3-0-0',
 'gfw-google-827.alerts.segment_info.2-5',
 'gfw-google-827.alerts.segment_info.3-0-0',
 'gfw-google-827.alerts.segment_vessel.2-5',
 'gfw-google-827.alerts.segment_vessel.3-0-0',
 'gfw-google-827.alerts.segs_activity.2-5',
 'gfw-google-827.alerts.segs_activity.3-0-0',
 'gfw-google-827.alerts.segs_activity_daily.2-5',
 'gfw-google-827.alerts.segs_activity_daily.3-0-0',
 'gfw-google-827.alerts.ssvids_identities.2-5',
 'gfw-google-827.alerts.ssvids_identities.3-0-0',
 'gfw-google-827.alerts.ssvids_identities_daily.2-5',
 'gfw-google-827.alerts.ssvids_identities_daily.3-0-0',
 'gfw-google-827.alerts.stats_daily.2-5',
 'gfw-google-827.alerts.stats_daily.3-0-0',
 'gfw-google-827.alerts.vessel_info.2-5',
 'gfw-google-827.alerts.vessel_info.3-0-0',
 'gfw-google-8

In [13]:
for current_expectation_suite_name in [es for es in context.list_expectation_suite_names() if 'segs_activity_daily' in es and 'constraints' in es]:
    print(current_expectation_suite_name)
    current_expectation_suite=context.get_expectation_suite(current_expectation_suite_name)    
    current_expectation_suite_asset_name=current_expectation_suite.meta.get('asset_name')
    current_expectation_suite_datasource_name=current_expectation_suite.meta.get('datasource_name')
    current_expectation_suite_version_number=current_expectation_suite.meta.get('version_number')

    gx_asset=gx_datasource.get_asset(current_expectation_suite_asset_name)
    gx_splitter=gx_asset.splitter
    if gx_splitter is not None:
        DATE_PARTITION_COLUMN=gx_splitter.column_name
        br_options={DATE_PARTITION_COLUMN: '2023-04-01'}
    else:
        br_options={}
    gx_br = gx_asset.build_batch_request(br_options)
    gx_batches = gx_datasource.get_batch_list_from_batch_request(gx_br)
    gx_validator = context.get_validator_using_batch_list(current_expectation_suite, gx_batches)

    gx_validator.expect_column_values_to_be_unique('seg_id')
    gx_validator.expect_column_values_to_not_be_null('seg_id')

    hours_cols = [hours_col for hours_col in gx_validator.columns() if 'hour' in hours_col and not '.' in hours_col]
    for current_hours_col in hours_cols:
        gx_validator.expect_column_values_to_be_between(
            column=current_hours_col, 
            min_value=0, 
            max_value=48, 
            strict_max=True
        )
        if current_hours_col != 'hours':
            gx_validator.expect_column_pair_values_a_to_be_greater_than_b(
                'hours', current_hours_col, or_equal=True
            )

    gx_validator.save_expectation_suite(discard_failed_expectations=False)

DEBUG:great_expectations.datasource.fluent.fluent_base_model:SQLDatasource.__fields_set__ assets added
INFO:great_expectations.datasource.fluent.fluent_base_model:SQLDatasource.dict() - substituting config values


gfw-google-827.constraints.segs_activity_daily.2-5


DEBUG:urllib3.connectionpool:https://bigquery.googleapis.com:443 "POST /bigquery/v2/projects/world-fishing-827/jobs?prettyPrint=false HTTP/1.1" 200 None
DEBUG:urllib3.connectionpool:https://bigquery.googleapis.com:443 "GET /bigquery/v2/projects/world-fishing-827/queries/c8709603-6529-4e46-98dd-215e6c0e4057?maxResults=0&location=US&prettyPrint=false HTTP/1.1" 200 None
DEBUG:urllib3.connectionpool:https://bigquery.googleapis.com:443 "GET /bigquery/v2/projects/world-fishing-827/jobs/c8709603-6529-4e46-98dd-215e6c0e4057?location=US&prettyPrint=false HTTP/1.1" 200 None
DEBUG:great_expectations.datasource.fluent.fluent_base_model:SQLDatasource.__fields_set__ assets added
INFO:great_expectations.datasource.fluent.fluent_base_model:SQLDatasource.dict() - substituting config values
DEBUG:urllib3.connectionpool:https://bigquery.googleapis.com:443 "POST /bigquery/v2/projects/world-fishing-827/jobs?prettyPrint=false HTTP/1.1" 200 None
DEBUG:urllib3.connectionpool:https://bigquery.googleapis.com:44

Calculating Metrics:   0%|          | 0/10 [00:00<?, ?it/s]

DEBUG:urllib3.connectionpool:https://bigquery.googleapis.com:443 "GET /bigquery/v2/projects/world-fishing-827/datasets/tech_great_expectations_temp_ttl_7d/tables/gx_temp_30feae90?prettyPrint=false HTTP/1.1" 200 None
  sqlalchemy.util.warn(

  sqlalchemy.util.warn(

DEBUG:great_expectations.execution_engine.sqlalchemy_execution_engine:Attempting query SELECT count(*) AS "table.row_count" 
FROM gx_temp_30feae90
DEBUG:urllib3.connectionpool:https://bigquery.googleapis.com:443 "POST /bigquery/v2/projects/world-fishing-827/jobs?prettyPrint=false HTTP/1.1" 200 None
DEBUG:urllib3.connectionpool:https://bigquery.googleapis.com:443 "GET /bigquery/v2/projects/world-fishing-827/queries/968fd6a4-c171-45b3-9947-8d6a06a3646c?maxResults=0&location=US&prettyPrint=false HTTP/1.1" 200 None
DEBUG:great_expectations.execution_engine.sqlalchemy_execution_engine:SqlAlchemyExecutionEngine computed 1 metrics on domain_id 193ba3158932159285fb120d0c39e1da
DEBUG:urllib3.connectionpool:https://bigquery.googleapis

...

DEBUG:urllib3.connectionpool:https://bigquery.googleapis.com:443 "GET /bigquery/v2/projects/world-fishing-827/datasets/tech_great_expectations_temp_ttl_7d/tables/gx_temp_6d865300?prettyPrint=false HTTP/1.1" 200 None
DEBUG:great_expectations.execution_engine.sqlalchemy_execution_engine:Attempting query SELECT count(*) AS "table.row_count" 
FROM gx_temp_6d865300
DEBUG:urllib3.connectionpool:https://bigquery.googleapis.com:443 "POST /bigquery/v2/projects/world-fishing-827/jobs?prettyPrint=false HTTP/1.1" 200 None
DEBUG:urllib3.connectionpool:https://bigquery.googleapis.com:443 "GET /bigquery/v2/projects/world-fishing-827/queries/acb29759-3079-49ec-bbef-0566c63a0a2e?maxResults=0&location=US&prettyPrint=false HTTP/1.1" 200 None
DEBUG:great_expectations.execution_engine.sqlalchemy_execution_engine:SqlAlchemyExecutionEngine computed 1 metrics on domain_id dbc862eb3bfc23e66bc15103eab2a7c2
DEBUG:urllib3.connectionpool:https://bigquery.googleapis.com:443 "POST /bigquery/v2/projects/world-fishing

Calculating Metrics:   0%|          | 0/11 [00:00<?, ?it/s]

DEBUG:urllib3.connectionpool:https://bigquery.googleapis.com:443 "GET /bigquery/v2/projects/world-fishing-827/datasets/tech_great_expectations_temp_ttl_7d/tables/gx_temp_6d865300?prettyPrint=false HTTP/1.1" 200 None
DEBUG:great_expectations.execution_engine.sqlalchemy_execution_engine:Attempting query SELECT count(*) AS "table.row_count" 
FROM gx_temp_6d865300
DEBUG:urllib3.connectionpool:https://bigquery.googleapis.com:443 "POST /bigquery/v2/projects/world-fishing-827/jobs?prettyPrint=false HTTP/1.1" 200 None
DEBUG:urllib3.connectionpool:https://bigquery.googleapis.com:443 "GET /bigquery/v2/projects/world-fishing-827/queries/fe035248-dcaa-4ea3-835b-87382b2092b4?maxResults=0&location=US&prettyPrint=false HTTP/1.1" 200 None
DEBUG:great_expectations.execution_engine.sqlalchemy_execution_engine:SqlAlchemyExecutionEngine computed 1 metrics on domain_id dbc862eb3bfc23e66bc15103eab2a7c2
DEBUG:urllib3.connectionpool:https://bigquery.googleapis.com:443 "POST /bigquery/v2/projects/world-fishing

Calculating Metrics:   0%|          | 0/8 [00:00<?, ?it/s]

DEBUG:urllib3.connectionpool:https://bigquery.googleapis.com:443 "GET /bigquery/v2/projects/world-fishing-827/datasets/tech_great_expectations_temp_ttl_7d/tables/gx_temp_6d865300?prettyPrint=false HTTP/1.1" 200 None
DEBUG:great_expectations.execution_engine.sqlalchemy_execution_engine:Attempting query SELECT count(*) AS "table.row_count" 
FROM gx_temp_6d865300
DEBUG:urllib3.connectionpool:https://bigquery.googleapis.com:443 "POST /bigquery/v2/projects/world-fishing-827/jobs?prettyPrint=false HTTP/1.1" 200 None
DEBUG:urllib3.connectionpool:https://bigquery.googleapis.com:443 "GET /bigquery/v2/projects/world-fishing-827/queries/384080f1-3a6b-4487-a95d-3620c8ebd8b1?maxResults=0&location=US&prettyPrint=false HTTP/1.1" 200 None
DEBUG:great_expectations.execution_engine.sqlalchemy_execution_engine:SqlAlchemyExecutionEngine computed 1 metrics on domain_id dbc862eb3bfc23e66bc15103eab2a7c2
DEBUG:urllib3.connectionpool:https://bigquery.googleapis.com:443 "POST /bigquery/v2/projects/world-fishing